# Open in Colab
<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ai-agents-the-definitive-guide/blob/main/CH10/ch10_memory_no_memory.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# About the Notebook


This notebook demonstrates the difference between running a LangGraph chatbot with and without checkpointed memory.

The graph contains a single chatbot node that sends the current message history to an LLM through the OpenRouter-compatible `ChatOpenAI` interface. Two versions of the same graph are compiled: one without a checkpointer and one with an `InMemorySaver` checkpointer.

In the first run, the graph receives each user message independently, so it does not retain the user’s name across invocations. In the second run, both invocations use the same `thread_id`, allowing LangGraph to restore the prior conversation state from memory. As a result, the chatbot can answer the follow-up question based on information provided in the previous turn.

The example illustrates the role of thread-level checkpointing in agent memory and shows why persistent state must be explicitly configured when building conversational or multi-turn agent systems.


In [ ]:
import os
from dotenv import load_dotenv


load_dotenv()

OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

In [ ]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.0 MB/s eta 0:00:00


In [ ]:
from typing import Annotated, TypedDict
import operator

from langchain_core.messages import AnyMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


llm = ChatOpenAI(model="gpt-5.4-nano", base_url="https://openrouter.ai/api/v1", temperature=0, api_key=OPENROUTER_API_KEY)


In [ ]:
# pip install -U langgraph langchain openai



# ---------------------------
# Define state
# ---------------------------
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


# ---------------------------
#  Define node
# ---------------------------
def chatbot(state: AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


# ---------------------------
# Build graph
# ---------------------------
builder = StateGraph(AgentState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)


# Version A: no memory
graph_without_memory = builder.compile()

# Version B: with memory
checkpointer = InMemorySaver()
graph_with_memory = builder.compile(checkpointer=checkpointer)


# ---------------------------
# Helper to print output
# ---------------------------
def show_last_ai_message(result: dict) -> None:
    last_message = result["messages"][-1]
    print(f"AI: {last_message.content}\n")


# ---------------------------
#  Run without memory
# ---------------------------
print("WITHOUT MEMORY")
print("=" * 60)

result = graph_without_memory.invoke(
    {
        "messages": [
            HumanMessage(content="My name is Nicole. I am writing a chapter about memory in AI agents. You don't need to give me tips, just ackknowledge it.")
        ]
    }
)
show_last_ai_message(result)

result = graph_without_memory.invoke(
    {
        "messages": [
            HumanMessage(content="What is my name?")
        ]
    }
)
show_last_ai_message(result)


# ---------------------------
# Run with memory
# ---------------------------
print("WITH MEMORY")
print("=" * 60)

config = {"configurable": {"thread_id": "demo-thread-1"}}

result = graph_with_memory.invoke(
    {
        "messages": [
            HumanMessage(content="My name is Nicole. I am writing a chapter about memory in AI agents.")
        ]
    },
    config=config,
)
show_last_ai_message(result)

result = graph_with_memory.invoke(
    {
        "messages": [
            HumanMessage(content="What is my name?")
        ]
    },
    config=config,
)
show_last_ai_message(result)

WITHOUT MEMORY
AI: Hi Nicole — acknowledged. Good luck writing your chapter on memory in AI agents.

AI: I don’t know your name from this chat. If you tell me what you’d like me to call you, I’ll use it.

WITH MEMORY
AI: Hi Nicole—nice to meet you! If you’re writing a chapter on memory in AI agents, I can help you shape the content, outline the chapter, or draft sections.

A few quick questions to tailor it:
1. **Audience level:** is this for general readers, undergraduate, or graduate/professional?
2. **Scope:** do you mean *agent memory* broadly (retrieval, long-term/short-term, planning), or mainly *neural memory* (transformers, external memory, RNN/attention)?
3. **Angle:** are you emphasizing **technical mechanisms** (e.g., RAG, vector DBs, episodic memory) or **systems/engineering** (latency, scaling, evaluation), or both?
4. **Length target:** roughly how many pages/words?

Meanwhile, here are a couple of strong chapter section structures you can choose from:

### Option A: Conc